# Clasificare cu cei mai apropiați vecini

Algoritmul k-NN clasifică un exemplu nou consultând cele mai asemănătoare exemple din setul de antrenare. Este simplu, dar scoate la iveală idei esențiale: reprezentarea datelor, distanța, normalizarea și evaluarea.

## 1. Problema și datele

Vom clasifica observații în două categorii folosind două trăsături numerice. O observație este un punct $x=(x_1,x_2)$, iar eticheta $y$ este `0` sau `1`.

> Într-un proiect real, numele trăsăturilor și modul de colectare sunt la fel de importante ca algoritmul.

In [1]:
X_train = [
    (1.0, 1.2), (1.3, 0.8), (1.8, 1.6), (2.0, 1.1),
    (4.8, 5.2), (5.1, 4.7), (5.5, 5.4), (4.6, 4.5),
]
y_train = [0, 0, 0, 0, 1, 1, 1, 1]

print("Exemple de antrenare:", len(X_train))
print("Clase:", sorted(set(y_train)))

Exemple de antrenare: 8
Clase: [0, 1]


## 2. Distanța euclidiană

Pentru două puncte, distanța este

$$d(a,b)=\sqrt{(a_1-b_1)^2+(a_2-b_2)^2}. $$

Rădăcina pătrată nu schimbă ordinea distanțelor, astfel încât putem compara direct distanțele la pătrat.

In [2]:
def distanta_patrat(a, b):
    return sum((xa - xb) ** 2 for xa, xb in zip(a, b))

punct_nou = (1.4, 1.1)
vecini = sorted(
    (distanta_patrat(punct_nou, x), eticheta)
    for x, eticheta in zip(X_train, y_train)
)
print("Distanțe ordonate:", vecini[:4])

Distanțe ordonate: [(0.17, 0), (0.82, 0), (1.37, 0), (1.62, 0)]


## 3. Votul majoritar

Alegem primii `k` vecini și întoarcem clasa care apare cel mai des. Un `k` foarte mic este sensibil la zgomot; unul prea mare poate ignora structura locală. Folosim de regulă un număr impar pentru a reduce egalitățile.

In [3]:
from collections import Counter

def prezice_knn(X, y, punct, k=3):
    vecini = sorted(
        (distanta_patrat(punct, x), eticheta)
        for x, eticheta in zip(X, y)
    )[:k]
    voturi = Counter(eticheta for _, eticheta in vecini)
    return voturi.most_common(1)[0][0]

for punct in [(1.4, 1.1), (4.9, 4.9), (3.0, 3.0)]:
    print(f"{punct} → clasa {prezice_knn(X_train, y_train, punct)}")

(1.4, 1.1) → clasa 0
(4.9, 4.9) → clasa 1
(3.0, 3.0) → clasa 0


## 4. De ce normalizăm

Dacă o trăsătură variază între 0 și 1, iar alta între 0 și 100 000, a doua va domina distanța. Standardizarea transformă fiecare coloană astfel încât să aibă aproximativ media 0 și deviația standard 1.

Parametrii normalizării se calculează **numai pe setul de antrenare**, altfel informația din test se scurge în model.

In [4]:
def standardizeaza(X):
    coloane = list(zip(*X))
    medii = [sum(c) / len(c) for c in coloane]
    deviatii = [
        (sum((v - m) ** 2 for v in c) / len(c)) ** 0.5
        for c, m in zip(coloane, medii)
    ]
    transformat = [
        tuple((v - m) / s for v, m, s in zip(rand, medii, deviatii))
        for rand in X
    ]
    return transformat, medii, deviatii

X_scalat, medii, deviatii = standardizeaza(X_train)
[tuple(round(v, 2) for v in rand) for rand in X_scalat[:3]]

[(-1.18, -1.02), (-0.99, -1.24), (-0.68, -0.77)]

## 5. Evaluarea clasificării

Acuratețea nu spune ce fel de greșeli face modelul. Pentru două clase, matricea de confuzie numără adevărate pozitive (TP), adevărate negative (TN), false pozitive (FP) și false negative (FN).

- **precizia**: $TP/(TP+FP)$ — câte predicții pozitive au fost corecte;
- **recall**: $TP/(TP+FN)$ — câte exemple pozitive au fost găsite.

In [5]:
reale =    [0, 0, 1, 1]
prezise = [0, 1, 1, 1]

tp = sum(r == 1 and p == 1 for r, p in zip(reale, prezise))
tn = sum(r == 0 and p == 0 for r, p in zip(reale, prezise))
fp = sum(r == 0 and p == 1 for r, p in zip(reale, prezise))
fn = sum(r == 1 and p == 0 for r, p in zip(reale, prezise))

print("Acuratețe:", (tp + tn) / len(reale))
print("Precizie:", round(tp / (tp + fp), 2))
print("Recall:", tp / (tp + fn))

Acuratețe: 0.75
Precizie: 0.67
Recall: 1.0


## Experiment

Construiește un set de validare și compară valorile `k = 1, 3, 5, 7`. Pentru fiecare valoare raportează acuratețea și matricea de confuzie.

Răspunde apoi argumentat:

1. Ce se întâmplă când adaugi un exemplu etichetat greșit?
2. Cum se schimbă predicția punctului `(3, 3)`?
3. De ce alegerea lui `k` pe setul de test ar produce o evaluare optimistă?

**Extensie:** implementează vot ponderat, în care un vecin contribuie cu $1/(d+\varepsilon)$.